# Basic Prompt Structures

## Overview

How you *structure* a prompt matters as much as what you put in it. This notebook focuses on the two fundamental architectural patterns every prompt engineer needs to master: the **single-turn prompt** — a self-contained instruction that asks for everything in one shot — and the **multi-turn prompt** — a sequence of exchanges where each message builds on the last.

These are not just technical distinctions. They represent two fundamentally different models of how you communicate with an AI, and choosing the wrong one for a given task is one of the most common reasons prompts underperform.

**Tools used in this notebook:** `openai` · `langchain` · `python-dotenv`

---

## Motivation

Most people start with single-turn prompts because they mirror how we use search engines: one input, one output, done. That works for simple queries. It breaks down the moment your task requires context, nuance, or back-and-forth refinement.

Multi-turn prompts solve this — but they introduce their own complexity. How do you maintain context across exchanges? What happens when the conversation grows long? How do you structure a dialogue so the model stays on track rather than drifting?

This notebook builds the mental model and the practical toolkit for both approaches, so you can make a deliberate choice about which structure serves your task rather than defaulting to whichever one feels familiar.

---

## Key Components

**Single-turn prompts** are self-contained interactions — the prompt includes everything the model needs to respond, and the exchange ends there. They are fast, simple, and effective for well-defined tasks that don't require iteration.

**Multi-turn prompts** are structured conversations where meaning accumulates across messages. The model's previous responses become part of the context for everything that follows, enabling tasks that require clarification, refinement, or sequential reasoning.

**Prompt templates** are parameterized structures that separate the fixed logic of a prompt from its variable inputs. Rather than rewriting a prompt for every new use case, you define the pattern once and substitute values as needed — making your prompts reusable, testable, and scalable.

**Conversation chains** are LangChain's mechanism for managing multi-turn dialogue. They handle the bookkeeping of conversation history automatically, so you can focus on the content of each exchange rather than manually threading context from one message to the next.

---

## Method Details

The notebook moves from simple to complex: starting with bare single-turn API calls, introducing templates to make them reusable, then building out into multi-turn conversations and managed conversation chains. Each section pairs a conceptual explanation with a working code example, and key sections include direct comparisons — the same task handled with and without structure — so the practical difference is concrete and observable rather than theoretical.

## Setup

First, let's import the necessary libraries and set up our environment.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

from langchain_core.prompts import ChatPromptTemplate

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# 1. Load the .env file
# This automatically puts variables into os.environ
load_dotenv() 

# 2. Check if the key exists (Fail Fast)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing! Check your .env file.")

# 3. Initialize (LangChain automatically looks for the env var)
llm = ChatOpenAI(model="gpt-4o-mini")

## 1. Single-Turn Prompts

A single-turn prompt is the most atomic unit of interaction with a language model — one input, one response, done. There is no back-and-forth, no memory of what came before, and no opportunity to clarify after the fact. The model reads your prompt, generates a completion, and stops.

This simplicity is both its strength and its constraint. Single-turn prompts are fast, lightweight, and easy to deploy at scale — a single well-crafted template can power thousands of independent requests without any conversational overhead. But because the model has only one shot at understanding your intent, everything depends on how clearly and completely that intent is expressed upfront.

The practical implication: **what you don't say, the model will assume.** Every gap in your prompt is filled by the model's best statistical guess based on its training data. Sometimes that guess aligns with what you wanted. Often it doesn't — and the only way to close the gap is to make your instructions more explicit.

Single-turn prompts are the foundation of everything else in this curriculum. Before you can chain prompts together, assign roles, or build reasoning pipelines, you need to be able to write a single prompt that reliably produces what you want. That skill — writing complete, unambiguous, well-scoped instructions — is what this section builds.

In [ ]:
single_turn_prompt = "What are the three primary colors?"
print(llm.invoke(single_turn_prompt).content)